In [3]:

# Download the official AutoDock Vina 1.2.7 Linux executable
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64 -O /usr/local/bin/vina

# Make it executable
!chmod +x /usr/local/bin/vina

# Verify installation
!vina --version
!vina --help


AutoDock Vina v1.2.7
AutoDock Vina v1.2.7

Input:
  --receptor arg             rigid part of the receptor (PDBQT)
  --flex arg                 flexible side chains, if any (PDBQT)
  --ligand arg               ligand (PDBQT)
  --batch arg                batch directory or ligands (PDBQT)
  --scoring arg (=vina)      scoring function (ad4, vina or vinardo)

Search space (required):
  --maps arg                 affinity maps for the autodock4.2 (ad4) or vina 
                             scoring function
  --center_x arg             X coordinate of the center (Angstrom)
  --center_y arg             Y coordinate of the center (Angstrom)
  --center_z arg             Z coordinate of the center (Angstrom)
  --size_x arg               size in the X dimension (Angstrom)
  --size_y arg               size in the Y dimension (Angstrom)
  --size_z arg               size in the Z dimension (Angstrom)
  --autobox                  set maps dimensions based on input ligand(s) (for 
                    

In [4]:

import os
import requests

# Create project directories
os.makedirs("tryptanthrin_docking/structures", exist_ok=True)
os.makedirs("tryptanthrin_docking/ligands", exist_ok=True)
os.makedirs("tryptanthrin_docking/results", exist_ok=True)

# Download EGFR structure
pdb_url = "https://files.rcsb.org/download/1M17.pdb"
pdb_path = "tryptanthrin_docking/structures/EGFR_1M17.pdb"

response = requests.get(pdb_url)
response.raise_for_status()

with open(pdb_path, "wb") as f:
    f.write(response.content)

print("Download successful!")
print("File:", pdb_path)
print("Size:", round(os.path.getsize(pdb_path) / 1024, 2), "KB")


Download successful!
File: tryptanthrin_docking/structures/EGFR_1M17.pdb
Size: 246.8 KB


In [7]:

!pip install -q biopython


In [8]:

import Bio
from Bio.PDB import PDBParser

print("Biopython version:", Bio.__version__)
print("PDBParser imported successfully!")


Biopython version: 1.88
PDBParser imported successfully!


In [9]:

parser = PDBParser(QUIET=True)

structure = parser.get_structure(
    "EGFR",
    "tryptanthrin_docking/structures/EGFR_1M17.pdb"
)

for model in structure:
    for chain in model:
        protein_residues = [
            r for r in chain.get_residues()
            if r.id[0] == " "
        ]
        print(
            "Chain:", chain.id,
            "| Protein residues:", len(protein_residues)
        )


Chain: A | Protein residues: 312


In [10]:

from Bio.PDB import PDBParser
from collections import Counter

pdb_path = "tryptanthrin_docking/structures/EGFR_1M17.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("EGFR", pdb_path)

for model in structure:
    hetero_residues = []

    for chain in model:
        for residue in chain:
            if residue.id[0].strip():
                hetero_residues.append(
                    (chain.id, residue.resname, residue.id[1])
                )

    print("Non-protein residues:")
    for item in hetero_residues:
        print(item)

    print("\nResidue counts:")
    print(Counter(x[1] for x in hetero_residues))


Non-protein residues:
('A', 'AQ4', 999)
('A', 'HOH', 1)
('A', 'HOH', 2)
('A', 'HOH', 3)
('A', 'HOH', 4)
('A', 'HOH', 5)
('A', 'HOH', 6)
('A', 'HOH', 7)
('A', 'HOH', 8)
('A', 'HOH', 9)
('A', 'HOH', 10)
('A', 'HOH', 11)
('A', 'HOH', 12)
('A', 'HOH', 13)
('A', 'HOH', 14)
('A', 'HOH', 15)
('A', 'HOH', 16)
('A', 'HOH', 17)
('A', 'HOH', 18)
('A', 'HOH', 19)
('A', 'HOH', 20)

Residue counts:
Counter({'HOH': 20, 'AQ4': 1})


In [11]:

import numpy as np
from Bio.PDB import PDBParser

pdb_path = "tryptanthrin_docking/structures/EGFR_1M17.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("EGFR", pdb_path)

model = structure[0]
ligand = model["A"][("H_AQ4", 999, " ")]

coordinates = np.array([
    atom.coord for atom in ligand.get_atoms()
])

center = coordinates.mean(axis=0)

print("Ligand atoms:", len(coordinates))
print("\nDocking grid centre:")
print("Center X:", round(center[0], 3))
print("Center Y:", round(center[1], 3))
print("Center Z:", round(center[2], 3))

print("\nGrid dimensions (initial):")
print("Size X: 22 Å")
print("Size Y: 22 Å")
print("Size Z: 22 Å")


Ligand atoms: 29

Docking grid centre:
Center X: 22.014
Center Y: 0.253
Center Z: 52.794

Grid dimensions (initial):
Size X: 22 Å
Size Y: 22 Å
Size Z: 22 Å


In [12]:

import importlib.util
import shutil

packages = ["meeko", "rdkit", "Bio"]

for package in packages:
    print(package, ":",
          "Available" if importlib.util.find_spec(package) else "Not installed")

print("\nMeeko receptor tool:", shutil.which("mk_prepare_receptor.py"))
print("Meeko ligand tool:", shutil.which("mk_prepare_ligand.py"))


meeko : Not installed
rdkit : Not installed
Bio : Available

Meeko receptor tool: None
Meeko ligand tool: None


In [13]:

%pip install -q rdkit meeko


In [15]:

%pip install -q gemmi


In [16]:

import os
import rdkit
import meeko
import gemmi
import Bio
import shutil

print("RDKit:", rdkit.__version__)
print("Meeko:", meeko.__version__ if hasattr(meeko, "__version__") else "Available")
print("Gemmi:", gemmi.__version__)
print("Biopython:", Bio.__version__)
print("Vina:", shutil.which("vina"))

print("PDB exists:",
      os.path.exists("tryptanthrin_docking/structures/EGFR_1M17.pdb"))


RDKit: 2026.09.2
Meeko: 0.8.0
Gemmi: 0.7.5
Biopython: 1.88
Vina: /usr/local/bin/vina
PDB exists: True


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:

!python -m pip show meeko
!command -v mk_prepare_receptor.py || true
!command -v mk_prepare_ligand.py || true
!python -m meeko.cli.mk_prepare_receptor --help


Name: meeko
Version: 0.8.0
Summary: Python package for preparing small molecule for docking
Home-page: https://github.com/forlilab/meeko
Author: Forli Lab
Author-email: forli@scripps.edu
License: LGPL-2.1
Location: /usr/local/lib/python3.13/dist-packages
Requires: 
Required-by: 
/usr/local/bin/mk_prepare_receptor.py
/usr/local/bin/mk_prepare_ligand.py
usage: mk_prepare_receptor.py [-h] [--read_pdb PDB_FILENAME]
                              [--read_json JSON_FILENAME]
                              [--read_pqr PQR_FILENAME] [-i MACROMOL_FILENAME]
                              [-o OUTPUT_BASENAME] [-p [PDBQT_FILENAME ...]]
                              [-j [JSON_FILENAME ...]]
                              [--write_pdb [PDB_FILENAME ...]]
                              [--ignore_https_cert] [-g [GPF_FILENAME ...]]
                              [-v [VINA_BOX_FILENAME ...]]
                              [--debug_fn DEBUG_FN] [-n SET_TEMPLATE]
                              [-d DELETE_RESIDUE

In [18]:

# Prepare EGFR receptor using Meeko 0.8.0

!mk_prepare_receptor.py \
    --read_pdb tryptanthrin_docking/structures/EGFR_1M17.pdb \
    -o tryptanthrin_docking/structures/EGFR \
    -p \
    --box_center 22.014 0.253 52.794 \
    --box_size 22 22 22


Template padding will be used.
 gasteiger charges will be read from template file 
 
/usr/local/lib/python3.13/dist-packages/meeko/polymer.py:1237: RuntimeWarning: Input residues {'A:999': 'AQ4'} not in residue_templates

  warnings.warn(err, RuntimeWarning)
/usr/local/lib/python3.13/dist-packages/meeko/polymer.py:1238: RuntimeWarning: Trying to resolve unknown residues by building chemical templates... 
  warnings.warn("Trying to resolve unknown residues by building chemical templates... ", RuntimeWarning)
Molecule doesn't contain matching atoms for acidic_proton_loc:{'[H][O][CX3](=O)': 0, '[H][O][SX4](=O)': 0, '[H][O][SX3](=O)': 0, '[H][O][PX4](=O)': 0, '[H][O][PX3](=O)': 0, '[H][O][CX3](=S)': 0, '[H][O][SX4](=S)': 0, '[H][SX2][a]': 0}-> deprotonate returning original mol... 

Error: Creation of data structure for receptor failed.

Details:
- Residues with alternate location: ['A:751', 'A:831']
Either specify an altloc for each with option wanted_altloc
or a general default altloc wi

In [19]:

import os

folder = "tryptanthrin_docking/structures"

for filename in sorted(os.listdir(folder)):
    path = os.path.join(folder, filename)
    if os.path.isfile(path):
        print(filename, "—", round(os.path.getsize(path) / 1024, 2), "KB")


EGFR_1M17.pdb — 246.8 KB


In [20]:

!mk_prepare_receptor.py \
    --read_pdb tryptanthrin_docking/structures/EGFR_1M17.pdb \
    -o tryptanthrin_docking/structures/EGFR \
    -p \
    --box_center 22.014 0.253 52.794 \
    --box_size 22 22 22


Template padding will be used.
 gasteiger charges will be read from template file 
 
/usr/local/lib/python3.13/dist-packages/meeko/polymer.py:1237: RuntimeWarning: Input residues {'A:999': 'AQ4'} not in residue_templates

  warnings.warn(err, RuntimeWarning)
/usr/local/lib/python3.13/dist-packages/meeko/polymer.py:1238: RuntimeWarning: Trying to resolve unknown residues by building chemical templates... 
  warnings.warn("Trying to resolve unknown residues by building chemical templates... ", RuntimeWarning)
Molecule doesn't contain matching atoms for acidic_proton_loc:{'[H][O][CX3](=O)': 0, '[H][O][SX4](=O)': 0, '[H][O][SX3](=O)': 0, '[H][O][PX4](=O)': 0, '[H][O][PX3](=O)': 0, '[H][O][CX3](=S)': 0, '[H][O][SX4](=S)': 0, '[H][SX2][a]': 0}-> deprotonate returning original mol... 

Error: Creation of data structure for receptor failed.

Details:
- Residues with alternate location: ['A:751', 'A:831']
Either specify an altloc for each with option wanted_altloc
or a general default altloc wi

In [21]:

from Bio.PDB import PDBParser

pdb_path = "tryptanthrin_docking/structures/EGFR_1M17.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("EGFR", pdb_path)

chain = structure[0]["A"]

for residue_number in [751, 831]:
    residue = chain[residue_number]

    print(f"\nResidue: {residue.resname} {residue_number}")

    for atom in residue:
        if atom.is_disordered():
            print(
                atom.get_name(),
                "Alternate locations:",
                [(a.get_altloc(), a.get_occupancy())
                 for a in atom.disordered_get_list()]
            )



Residue: CYS 751
N Alternate locations: [('A', 0.5), ('B', 0.5)]
CA Alternate locations: [('A', 0.5), ('B', 0.5)]
C Alternate locations: [('A', 0.5), ('B', 0.5)]
O Alternate locations: [('A', 0.5), ('B', 0.5)]
CB Alternate locations: [('A', 0.5), ('B', 0.5)]
SG Alternate locations: [('A', 0.5), ('B', 0.5)]

Residue: ASP 831
N Alternate locations: [('A', 0.5), ('B', 0.5)]
CA Alternate locations: [('A', 0.5), ('B', 0.5)]
C Alternate locations: [('A', 0.5), ('B', 0.5)]
O Alternate locations: [('A', 0.5), ('B', 0.5)]
CB Alternate locations: [('A', 0.5), ('B', 0.5)]
CG Alternate locations: [('A', 0.5), ('B', 0.5)]
OD1 Alternate locations: [('A', 0.5), ('B', 0.5)]
OD2 Alternate locations: [('A', 0.5), ('B', 0.5)]


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:

from Bio.PDB import PDBParser, PDBIO, Select
import os

input_pdb = "tryptanthrin_docking/structures/EGFR_1M17.pdb"
output_pdb = "tryptanthrin_docking/structures/EGFR_clean.pdb"

class EGFRReceptorSelect(Select):

    def accept_chain(self, chain):
        return chain.id == "A"

    def accept_residue(self, residue):
        # Keep standard protein residues only
        return residue.id[0] == " "

    def accept_atom(self, atom):
        altloc = atom.get_altloc()

        # Keep blank atoms and alternate location A
        return altloc in (" ", "A")

parser = PDBParser(QUIET=True)
structure = parser.get_structure("EGFR", input_pdb)

io = PDBIO()
io.set_structure(structure)
io.save(output_pdb, EGFRReceptorSelect())

print("Cleaned receptor saved:", output_pdb)
print("File size:", round(os.path.getsize(output_pdb)/1024, 2), "KB")


Cleaned receptor saved: tryptanthrin_docking/structures/EGFR_clean.pdb
File size: 197.6 KB


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [23]:

from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
cleaned = parser.get_structure(
    "EGFR_clean",
    "tryptanthrin_docking/structures/EGFR_clean.pdb"
)

chain = cleaned[0]["A"]

print("Protein residues:", len(list(chain.get_residues())))

for number in [751, 831]:
    residue = chain[number]
    print(
        residue.resname,
        number,
        "alternate locations:",
        sorted(set(
            atom.get_altloc() for atom in residue
        ))
    )


Protein residues: 312
CYS 751 alternate locations: ['A']
ASP 831 alternate locations: ['A']


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:

!mk_prepare_receptor.py \
    --read_pdb tryptanthrin_docking/structures/EGFR_clean.pdb \
    -o tryptanthrin_docking/structures/EGFR \
    -p \
    --box_center 22.014 0.253 52.794 \
    --box_size 22 22 22


Template padding will be used.
 gasteiger charges will be read from template file 
 

Error: Creation of data structure for receptor failed.

Details:
- Residues with alternate location: ['A:751', 'A:831']
Either specify an altloc for each with option wanted_altloc
or a general default altloc with option default_altloc.
Recommendations:
1. (for batch processing) Use -a/--allow_bad_res to automatically remove residues
that do not match templates, and --default_altloc to set
a default altloc variant. Use these at your own risk.

2. (processing individual structure) Inspecting and fixing the input structure is recommended.
Use --wanted_altloc to set variants for specific residues.




In [25]:

import os

folder = "tryptanthrin_docking/structures"

for filename in sorted(os.listdir(folder)):
    if filename.startswith("EGFR"):
        path = os.path.join(folder, filename)
        print(filename, round(os.path.getsize(path)/1024, 2), "KB")


EGFR_1M17.pdb 246.8 KB
EGFR_clean.pdb 197.6 KB


In [26]:

input_pdb = "tryptanthrin_docking/structures/EGFR_clean.pdb"

with open(input_pdb, "r") as f:
    lines = f.readlines()

corrected = []

for line in lines:
    if line.startswith(("ATOM  ", "HETATM")):
        altloc = line[16]

        # Remove alternate conformer B
        if altloc == "B":
            continue

        # Keep conformer A, but make it the standard location
        if altloc == "A":
            line = line[:16] + " " + line[17:]

    corrected.append(line)

with open(input_pdb, "w") as f:
    f.writelines(corrected)

print("PDB alternate locations corrected.")


PDB alternate locations corrected.
